# ¿Cambian los resultados con más usuarios o con otra muestra?

Proyecto Integrador — Ciencia de Datos · UTN FRM · 2026 · Grupo 5K09-05

El DAG `movielens_ingest` elige al azar (random_state = 42) usuarios de MovieLens 32M con al menos
100 ratings. Para las entregas usamos **5.000**. Corrimos el mismo DAG con **20.000**, sin cambiar
nada más (tope de 500 ratings por usuario, 10.000 películas consultadas a TMDB), y comparamos.

Después le agregamos al DAG un parámetro **Semilla** (42 por defecto, la de las entregas) y lo
corrimos con **otras tres semillas** (7, 123 y 2026), 5.000 usuarios cada una, para ver si otra
muestra de los 32 millones de ratings cambia los resultados.

**Preguntas:** ¿cuatro veces más usuarios cambia los datos o los grupos? ¿Y otra muestra al azar?

## Hoja de ruta

| Parte | Qué comparamos |
|---|---|
| **1** | Cuántos datos trae cada corrida |
| **2** | Si la muestra tiene la misma forma |
| **3** | Cuánto cubren las 10.000 películas de TMDB |
| **4** | El modelo: grupos, separación y estabilidad |
| **5** | Si los grupos describen los mismos gustos |
| **6** | Otras semillas: cuatro muestras distintas de 5.000 usuarios |
| **7** | Conclusión |

Las salidas de cada corrida están dentro del proyecto de Airflow, en `include/corrida_5000_usuarios/`
(la corrida de las entregas), `include/corrida_20000_usuarios/` y `include/corrida_semilla_*/`.

In [1]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import altair as alt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score

from modelado import SEMILLA, MIN_RATINGS, entrenar_desde_csv
from historia import GENERO_ES   # nombres de los géneros en castellano

alt.data_transformers.disable_max_rows()
pd.set_option("display.width", 160)
QUEMADO, CHARTREUSE, NAVY, ELECTRICO, GRIS = "#8b3a26", "#c4a22c", "#3e3f66", "#3b47c9", "#80818f"

DAG = Path(r"D:\Sol Personal NO BORRAR¡¡¡¡¡¡\Desktop\airflow-movielens-editado\airflow-movielens\include")
CORRIDAS = {"5.000 usuarios": DAG / "corrida_5000_usuarios", "20.000 usuarios": DAG / "corrida_20000_usuarios"}
COLOR = alt.Scale(domain=list(CORRIDAS), range=[NAVY, QUEMADO])

## Parte 1 — Cuántos datos trae cada corrida

Para cada corrida leemos la tabla silver (una fila por usuario), los ratings elegidos y las películas
preparadas desde TMDB.

In [2]:
datos = {}
for nombre, carpeta in CORRIDAS.items():
    silver = pd.read_csv(carpeta / "silver" / "usuarios_generos.csv")
    ratings = pd.read_csv(carpeta / "intermediate" / "ratings_selected_users.csv", usecols=["userId", "movieId"])
    tmdb = pd.read_csv(carpeta / "intermediate" / "tmdb_movies_prepared.csv", usecols=["movieId", "genres"])
    con_genero = set(tmdb.loc[tmdb["genres"].fillna("") != "", "movieId"])
    datos[nombre] = {"silver": silver, "ratings": ratings, "con_genero": con_genero}

resumen = pd.DataFrame({
    nombre: {
        "usuarios": len(d["silver"]),
        "ratings (con el tope de 500)": len(d["ratings"]),
        "películas distintas": d["ratings"]["movieId"].nunique(),
        "películas con género de TMDB": len(d["con_genero"]),
        f"usuarios con {MIN_RATINGS} ratings o más (entran al modelo)":
            int((d["silver"]["cantidad_valoraciones"] >= MIN_RATINGS).sum()),
    } for nombre, d in datos.items()})
resumen["cambio"] = (resumen.iloc[:, 1] / resumen.iloc[:, 0]).map(lambda r: f"× {r:.2f}".replace(".", ","))
resumen

,5.000 usuarios,20.000 usuarios,cambio
usuarios,5000,20000,"× 4,00"
ratings (con el tope de 500),1283510,5139331,"× 4,00"
películas distintas,26183,41188,"× 1,57"
películas con género de TMDB,9926,9926,"× 1,00"
usuarios con 150 ratings o más (entran al modelo),3503,13865,"× 3,96"


## Parte 2 — ¿La muestra tiene la misma forma?

Las dos corridas eligen al azar entre los mismos 80.675 usuarios con 100 ratings o más, así que
deberían parecerse. Comparamos la cantidad de ratings por usuario, el promedio que ponen y la época
de lo que miran, cada corrida en % de sus propios usuarios.

In [3]:
largo = pd.concat([d["silver"].assign(corrida=n) for n, d in datos.items()], ignore_index=True)
largo["log_ratings"] = np.log10(largo["cantidad_valoraciones_total"])
grupo = alt.selection_point(fields=["corrida"], bind="legend")


def histograma(campo, titulo_eje, paso, formato=None, extension=None):
    eje = alt.Axis(labelExpr="format(pow(10, datum.value), ',.0f')") if formato == "log" else alt.Axis()
    # cada gráfico recibe sólo las columnas que usa (si no, el notebook guarda las 54 columnas)
    return alt.Chart(largo[["corrida", campo]]).transform_bin("b", campo, bin=alt.Bin(step=paso, extent=extension)).transform_aggregate(
        n="count()", groupby=["b", "b_end", "corrida"]).transform_joinaggregate(
        total="sum(n)", groupby=["corrida"]).transform_calculate(pct="datum.n / datum.total").mark_bar(
        binSpacing=0, stroke="#ffffff", strokeWidth=0.4).encode(
        x=alt.X("b:Q", title=titulo_eje, axis=eje), x2="b_end:Q",
        y=alt.Y("pct:Q", title="% de usuarios", stack=None, axis=alt.Axis(format="%")), y2=alt.datum(0),
        color=alt.Color("corrida:N", scale=COLOR, title=None, legend=alt.Legend(orient="top")),
        opacity=alt.condition(grupo, alt.value(0.5), alt.value(0.05)),
        tooltip=["corrida:N", alt.Tooltip("pct:Q", format=".1%")]
    ).add_params(grupo).properties(width=300, height=200)


(histograma("log_ratings", "Ratings por usuario (escala log)", 0.1, "log", [2, 4.5])
 | histograma("rating_promedio_general", "Rating promedio del usuario", 0.1, extension=[1, 5])
 ).properties(title=alt.Title("Las dos muestras tienen la misma forma",
                              subtitle="Histogramas superpuestos, cada uno en % de su corrida. Clic en la leyenda para dejar uno.",
                              anchor="start", fontSize=15)).configure_view(stroke=None)

alt.HConcatChart(...)

In [4]:
epocas = ["prop_antes_1980", "prop_1980s", "prop_1990s", "prop_2000s", "prop_2010s", "prop_2020_mas"]
comparacion = largo.groupby("corrida")[["cantidad_valoraciones_total", "rating_promedio_general"] + epocas].mean().T
comparacion.round(3)

corrida,20.000 usuarios,5.000 usuarios
cantidad_valoraciones_total,325.296,321.740
rating_promedio_general,3.623,3.616
prop_antes_1980,0.111,0.110
prop_1980s,0.120,0.120
prop_1990s,0.350,0.351
prop_2000s,0.282,0.283
prop_2010s,0.131,0.131
prop_2020_mas,0.006,0.006


## Parte 3 — ¿Cuánto cubren las 10.000 películas de TMDB?

El DAG consulta a TMDB las 10.000 películas más vistas de la muestra. Con más usuarios aparecen más
películas distintas, así que esas 10.000 podrían cubrir una parte menor de los ratings.

In [5]:
filas = []
for nombre, d in datos.items():
    conteo = d["ratings"]["movieId"].value_counts().to_numpy()
    acumulado = np.cumsum(conteo) / conteo.sum()
    puntos = np.unique(np.round(np.logspace(0, np.log10(len(conteo)), 250)).astype(int))
    filas.append(pd.DataFrame({"corrida": nombre, "peliculas": puntos, "cubierto": acumulado[puntos - 1]}))
    print(f"{nombre}: el {d['ratings']['movieId'].isin(d['con_genero']).mean():.1%} de los ratings "
          f"tiene género de TMDB ({len(conteo):,} películas distintas)".replace(",", "."))
cobertura = pd.concat(filas)

regla = alt.Chart(pd.DataFrame({"x": [10_000]})).mark_rule(color=GRIS, strokeDash=[4, 3]).encode(x="x:Q")
curvas = alt.Chart(cobertura).mark_line(strokeWidth=2).encode(
    x=alt.X("peliculas:Q", title="Películas, de la más vista a la menos vista (escala log)", scale=alt.Scale(type="log")),
    y=alt.Y("cubierto:Q", title="Ratings cubiertos", axis=alt.Axis(format="%")),
    color=alt.Color("corrida:N", scale=COLOR, title=None, legend=alt.Legend(orient="top")),
    tooltip=["corrida:N", alt.Tooltip("peliculas:Q", format=","), alt.Tooltip("cubierto:Q", format=".1%")])
(regla + curvas).properties(width=560, height=280, title=alt.Title(
    "Las 10.000 películas más vistas siguen cubriendo casi todos los ratings",
    subtitle="La línea punteada marca las 10.000 películas que consulta el DAG.", anchor="start", fontSize=15)
).configure_view(stroke=None)

5.000 usuarios: el 97.6% de los ratings tiene género de TMDB (26.183 películas distintas)
20.000 usuarios: el 97.5% de los ratings tiene género de TMDB (41.188 películas distintas)


alt.LayerChart(...)

## Parte 4 — El modelo con cada corrida

Entrenamos el modelo de la Entrega 3 (`entrenar_desde_csv`, sin ningún cambio) con la tabla silver de
cada corrida y medimos lo mismo que en la entrega: cuántas componentes guarda la PCA, cuánto separan
los grupos (silhouette), cómo se reparten los usuarios y qué tan estables son los grupos (30 remuestreos
del 80 % de los usuarios).

In [6]:
modelos, espacios = {}, {}
for nombre, carpeta in CORRIDAS.items():
    m = entrenar_desde_csv(str(carpeta / "silver" / "usuarios_generos.csv"))
    modelos[nombre] = m
    D = m.entrenamiento
    espacios[nombre] = {
        "Géneros": (m.generos["preparar"].transform(D), m.etiquetas["grupo_genero"].to_numpy(), m.etiquetas["nombre_genero"]),
        "Épocas": (m.decadas["preparar"].transform(D), m.etiquetas["grupo_decada"].to_numpy(), m.etiquetas["nombre_decada"]),
    }


def estabilidad(Z, k=2, repeticiones=30):
    rng = np.random.default_rng(SEMILLA)
    completo = KMeans(n_clusters=k, random_state=SEMILLA, n_init=10).fit_predict(Z)
    aris = []
    for _ in range(repeticiones):
        idx = rng.choice(len(Z), size=int(0.8 * len(Z)), replace=False)
        parcial = KMeans(n_clusters=k, random_state=int(rng.integers(1_000_000)), n_init=10).fit(Z[idx]).predict(Z)
        aris.append(adjusted_rand_score(completo, parcial))
    return np.mean(aris)


filas = []
for nombre, esp in espacios.items():
    for espacio, (Z, grupo, etiqueta) in esp.items():
        reparto = etiqueta.value_counts(normalize=True)
        filas.append({"corrida": nombre, "espacio": espacio, "usuarios": len(Z), "componentes PCA": Z.shape[1],
                      "silhouette (k = 2)": silhouette_score(Z, grupo, sample_size=min(len(Z), 6000), random_state=SEMILLA),
                      "grupo más grande": f"{reparto.index[0]} ({reparto.iloc[0]:.0%})",
                      "estabilidad (ARI medio)": estabilidad(Z)})
modelo_tabla = pd.DataFrame(filas).set_index(["espacio", "corrida"]).sort_index()
modelo_tabla.round(3)

usuarios  componentes PCA  silhouette (k = 2)                     grupo más grande  estabilidad (ARI medio)
espacio corrida                                                                                                                     
Géneros 20.000 usuarios     13865                8               0.171  Animación, familia y aventura (61%)                    0.962
        5.000 usuarios       3503                8               0.175  Animación, familia y aventura (63%)                    0.962
Épocas  20.000 usuarios     13865                2               0.486              Cine de 2000–2010 (50%)                    0.992
        5.000 usuarios       3503                3               0.422       Cine noventero y clásico (50%)                    0.988

La silhouette con 20.000 usuarios se calcula sobre una muestra de 6.000 para que no tarde demasiado
(con 5.000 usa a todos). Con tantos usuarios, la diferencia que eso introduce es menor que la tercera
cifra decimal.

## Parte 5 — ¿Los grupos describen los mismos gustos?

Que los números se parezcan no alcanza: los grupos podrían haber cambiado de significado. Comparamos
la **afinidad media por género** de cada grupo en las dos corridas. Si los puntos de una corrida caen
sobre los de la otra, los grupos describen los mismos gustos.

In [7]:
filas = []
for nombre, m in modelos.items():
    D = m.entrenamiento
    pref = m.generos["preparar"]["features"].transform(D)
    pref.columns = [GENERO_ES.get(c.replace("pref_", ""), c) for c in pref.columns]
    medias = pref.groupby(m.etiquetas["nombre_genero"].to_numpy()).mean()
    for grupo_nombre, fila in medias.iterrows():
        for genero, valor in fila.items():
            filas.append({"corrida": nombre, "grupo": grupo_nombre, "genero": genero, "afinidad": valor})
afinidades = pd.DataFrame(filas)
orden = (afinidades[afinidades["corrida"] == "5.000 usuarios"].pivot(index="genero", columns="grupo", values="afinidad")
         .diff(axis=1).iloc[:, -1].sort_values().index.tolist())

alt.Chart(afinidades).mark_point(filled=False, size=90, strokeWidth=2).encode(
    y=alt.Y("genero:N", sort=orden, title=None),
    x=alt.X("afinidad:Q", title="Afinidad media del grupo (respecto del promedio propio)"),
    color=alt.Color("corrida:N", scale=COLOR, title=None, legend=alt.Legend(orient="top")),
    shape=alt.Shape("grupo:N", title=None, legend=alt.Legend(orient="top", labelLimit=0)),
    tooltip=["corrida", "grupo", "genero", alt.Tooltip("afinidad:Q", format="+.3f")]
).properties(width=520, height=440, title=alt.Title(
    "Los dos grupos de género describen los mismos gustos en las dos corridas",
    subtitle="Círculo y cuadrado: cada grupo. Navy: 5.000 usuarios; burnt orange: 20.000. Si se superponen, el grupo es el mismo.",
    anchor="start", fontSize=15)).configure_view(stroke=None)

alt.Chart(...)

In [8]:
# los usuarios que están en las dos corridas: ¿caen en el mismo grupo?
ids_comunes = modelos["5.000 usuarios"].etiquetas.index.intersection(modelos["20.000 usuarios"].etiquetas.index)
print(f"Usuarios modelados en las dos corridas: {len(ids_comunes):,}".replace(",", "."))
if len(ids_comunes) > 0:
    a = modelos["5.000 usuarios"].etiquetas.loc[ids_comunes]
    b = modelos["20.000 usuarios"].etiquetas.loc[ids_comunes]
    print(f"Mismo grupo de género: {(a['nombre_genero'] == b['nombre_genero']).mean():.1%}")
    print(f"Mismo grupo de época:  {(a['nombre_decada'] == b['nombre_decada']).mean():.1%}")

Usuarios modelados en las dos corridas: 3.503
Mismo grupo de género: 96.2%
Mismo grupo de época:  98.6%


## Parte 6 — Otras semillas

Cuatro corridas del DAG con 5.000 usuarios, cada una con su semilla. Son muestras distintas de los
80.675 usuarios con 100 ratings o más: cada par comparte sólo unos 300 usuarios, lo esperable por azar.
Para cada una entrenamos el mismo modelo y medimos lo mismo que en la Parte 4.

In [9]:
SEMILLAS = {"42 (entregas)": DAG / "corrida_5000_usuarios", "7": DAG / "corrida_semilla_7",
            "123": DAG / "corrida_semilla_123", "2026": DAG / "corrida_semilla_2026"}

ids = {n: set(pd.read_csv(c / "silver" / "usuarios_generos.csv", usecols=["userId"])["userId"]) for n, c in SEMILLAS.items()}
nombres = list(ids)
pd.DataFrame([[len(ids[a] & ids[b]) for b in nombres] for a in nombres], index=nombres, columns=nombres)

,42 (entregas),7,123,2026
42 (entregas),5000,302,333,305
7,302,5000,347,309
123,333,347,5000,325
2026,305,309,325,5000


In [10]:
filas, perfiles = [], []
for nombre, carpeta in SEMILLAS.items():
    silver = carpeta / "silver" / "usuarios_generos.csv"
    ratings = pd.read_csv(carpeta / "intermediate" / "ratings_selected_users.csv", usecols=["movieId"])
    tmdb = pd.read_csv(carpeta / "intermediate" / "tmdb_movies_prepared.csv", usecols=["movieId", "genres"])
    con_genero = tmdb.loc[tmdb["genres"].fillna("") != "", "movieId"]
    m = entrenar_desde_csv(str(silver))
    D = m.entrenamiento
    zg, zd = m.generos["preparar"].transform(D), m.decadas["preparar"].transform(D)
    # épocas con la cantidad de componentes fija, para separar el efecto de la PCA
    X = m.decadas["preparar"][:-1].transform(D)
    sil_fija = {}
    for k in (2, 3):
        Z = PCA(n_components=k, random_state=SEMILLA).fit_transform(X)
        sil_fija[k] = silhouette_score(Z, KMeans(n_clusters=2, random_state=SEMILLA, n_init=10).fit_predict(Z))
    var2 = np.cumsum(PCA().fit(X).explained_variance_ratio_)[1]
    filas.append({
        "semilla": nombre, "usuarios modelados": len(D), "películas distintas": ratings["movieId"].nunique(),
        "ratings con género TMDB": ratings["movieId"].isin(set(con_genero)).mean(),
        "silhouette géneros": silhouette_score(zg, m.etiquetas["grupo_genero"]),
        "componentes épocas": zd.shape[1], "varianza con 2 componentes": var2,
        "silhouette épocas (modelo)": silhouette_score(zd, m.etiquetas["grupo_decada"]),
        "silhouette épocas con 2": sil_fija[2], "silhouette épocas con 3": sil_fija[3],
        "% en animación y familia": (m.etiquetas["nombre_genero"] == "Animación, familia y aventura").mean(),
    })
    pref = m.generos["preparar"]["features"].transform(D)
    pref.columns = [GENERO_ES.get(c.replace("pref_", ""), c) for c in pref.columns]
    dif = pref.groupby(m.etiquetas["nombre_genero"].to_numpy()).mean()
    perfiles.append((dif.loc["Drama, crimen e histórico"] - dif.loc["Animación, familia y aventura"]).rename(nombre))
semillas = pd.DataFrame(filas).set_index("semilla")
semillas.T.round(3)

semilla,42 (entregas),7,123,2026
usuarios modelados,3503.000,3482.000,3477.000,3461.000
películas distintas,26183.000,25971.000,25192.000,26734.000
ratings con género TMDB,0.976,0.975,0.977,0.974
silhouette géneros,0.175,0.189,0.173,0.162
componentes épocas,3.000,2.000,2.000,3.000
varianza con 2 componentes,0.798,0.804,0.808,0.798
silhouette épocas (modelo),0.422,0.489,0.485,0.426
silhouette épocas con 2,0.479,0.489,0.485,0.485
silhouette épocas con 3,0.422,0.433,0.430,0.426
% en animación y familia,0.629,0.650,0.612,0.589


Para ver si los grupos significan lo mismo en las cuatro muestras, comparamos su **perfil**: para cada
género, cuánta más afinidad tiene el grupo de drama y crimen que el de animación y familia. Si las
cuatro corridas encuentran los mismos grupos, los cuatro perfiles coinciden.

In [11]:
perfil = pd.concat(perfiles, axis=1)
print("Correlación entre los perfiles de las cuatro muestras (1 = idénticos):")
print(perfil.corr().round(3))
largo_perfil = perfil.reset_index(names="genero").melt(id_vars="genero", var_name="semilla", value_name="diferencia")
orden = perfil.mean(axis=1).sort_values().index.tolist()
puntos = alt.Chart(largo_perfil).mark_point(filled=False, size=70, strokeWidth=1.8).encode(
    y=alt.Y("genero:N", sort=orden, title=None),
    x=alt.X("diferencia:Q", title="Afinidad del grupo de drama y crimen − la del de animación y familia"),
    color=alt.Color("semilla:N", title="Semilla", legend=alt.Legend(orient="top"),
                    scale=alt.Scale(domain=list(SEMILLAS), range=[NAVY, QUEMADO, CHARTREUSE, ELECTRICO])),
    tooltip=["semilla", "genero", alt.Tooltip("diferencia:Q", format="+.3f")])
cero = alt.Chart(pd.DataFrame({"x": [0]})).mark_rule(color=GRIS).encode(x="x:Q")
(cero + puntos).properties(width=520, height=420, title=alt.Title(
    "Las cuatro muestras encuentran los mismos dos grupos de género",
    subtitle="Cada punto es una semilla. Si caen juntos, el grupo significa lo mismo con cualquier muestra.",
    anchor="start", fontSize=15)).configure_view(stroke=None)

Correlación entre los perfiles de las cuatro muestras (1 = idénticos):
               42 (entregas)      7    123   2026
42 (entregas)          1.000  0.994  0.997  0.998
7                      0.994  1.000  0.998  0.995
123                    0.997  0.998  1.000  0.996
2026                   0.998  0.995  0.996  1.000


alt.LayerChart(...)

## Parte 7 — Conclusión

| Qué miramos | 5.000 usuarios | 20.000 usuarios | Qué significa |
|---|---|---|---|
| Ratings | 1.283.510 | 5.139.331 | Cuatro veces más, como era de esperar. |
| Películas distintas | 26.183 | 41.188 | Sólo 1,6 veces más: los usuarios nuevos repiten mucho las mismas películas. |
| Ratings con género de TMDB | 97,6 % | 97,5 % | Las 10.000 películas más vistas siguen alcanzando; no hace falta consultar más. |
| Forma de la muestra | — | — | Igual: mismo promedio, misma actividad, mismas épocas (diferencias en la tercera cifra). |
| Usuarios que entran al modelo | 3.503 | 13.865 | El 70 % en las dos. |
| Géneros: silhouette / estabilidad | 0,175 / 0,96 | 0,171 / 0,96 | Igual. La separación modesta es una propiedad de los gustos, no de la muestra chica. |
| Épocas: componentes de PCA | 3 | 2 | Cambia: con más usuarios, dos componentes ya pasan el 80 % de varianza. |
| Épocas: silhouette / estabilidad | 0,422 / 0,99 | 0,486 / 0,99 | Mejora la separación, en parte por usar una componente menos. |
| Mismo grupo para los mismos usuarios | — | 96 % (género), 99 % (época) | Los grupos son los mismos. |

**Un detalle del DAG:** la muestra de 5.000 está **dentro** de la de 20.000. El DAG elige al azar con la
misma semilla, así que los primeros 5.000 elegidos son los mismos; los 3.503 usuarios modelados en la
Entrega 3 están todos en la corrida nueva, y casi todos quedan en el mismo grupo.

**Otras semillas (Parte 6).** Cuatro muestras distintas de 5.000 usuarios, que comparten apenas unos
300 usuarios entre sí, dan lo mismo: el mismo 70 % entra al modelo, TMDB cubre entre el 97,4 y el
97,7 % de los ratings, la silhouette de géneros va de 0,16 a 0,19 y los perfiles de los grupos de
género coinciden casi perfecto (correlación de 0,994 o más).

**La silhouette de épocas no mejora con más usuarios.** Salta entre dos valores, 0,42 y 0,49, según la
PCA guarde 3 o 2 componentes. Y eso depende de un detalle: con 2 componentes la varianza explicada da
casi exactamente el 80 % que pedimos (entre 79,8 y 80,8 % según la muestra), así que el umbral cae de
un lado o del otro por azar. Con la misma cantidad de componentes, todas las muestras dan lo mismo
(0,48–0,49 con 2 y 0,42–0,43 con 3), con 5.000 o con 20.000 usuarios.

**Conclusión:** ni cuadruplicar los usuarios ni cambiar la muestra **cambian los resultados**. Los grupos
describen los mismos gustos con cualquier muestra, y la diferencia de silhouette en épocas es un efecto
del umbral del 80 % de la PCA, no de la cantidad de datos. **5.000 usuarios eran suficientes** para las
conclusiones de las entregas. Lo que sí conviene decir en la entrega es que la silhouette de épocas
está entre 0,42 y 0,49 según cuántas componentes se guarden, y que los grupos son los mismos en los dos
casos.